# MDU — Fəsil 20: Yekun layihə — Maşın öyrənməsi əsasında intellektual analiz sistemi

Bu notebook layihənin tam iş axınını nümayiş etdirir. Nümunə UCI Wine Quality datasetindən istifadə edir. Datasetin məqsədi şərabın fiziki-kimyəvi xüsusiyyətləri əsasında keyfiyyət balını təhlil etməkdir. Bu tədris nümunəsi kommersiya və ya ekspert qərarını əvəz etmir.

## 1. Mühitin hazırlanması və datasetin əldə edilməsi

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import sklearn, joblib, platform
print('Python:',platform.python_version(),'scikit-learn:',sklearn.__version__)

## 2. Real dataseti yükləyin və təsvir edin
OpenML-dəki wine-quality-red datasetindən istifadə olunur. İnternet bağlantısı tələb olunur. Datasetin versiyasını və OpenML səhifəsindəki lisenziya/mənbə qeydlərini layihə hesabatında yoxlayıb qeyd edin.

In [ ]:
wine = fetch_openml(name='wine-quality-red', version=1, as_frame=True)
df = wine.frame.copy()
df.head(), df.shape

In [ ]:
display(df.info())
display(df.describe().T)
print('Missing values:',df.isna().sum().sum())
print(df['quality'].value_counts().sort_index())

## 3. EDA — hədəf paylanması və korrelyasiya

In [ ]:
df['quality'] = pd.to_numeric(df['quality'], errors='coerce')
fig,ax=plt.subplots(1,2,figsize=(13,4))
df['quality'].value_counts().sort_index().plot(kind='bar',ax=ax[0],title='Quality paylanması')
sns.heatmap(df.corr(numeric_only=True),cmap='coolwarm',center=0,ax=ax[1])
ax[1].set_title('Korrelyasiya matrisi');plt.tight_layout()

## 4. Problemi formalaşdırın
Bu nümunədə keyfiyyət balı 6 və yuxarı olduqda `good=1`, əks halda `good=0` kimi ikili hədəf yaradılır. Bu hədd yalnız tədris nümunəsidir; real tədqiqatda domen əsaslandırması tələb olunur.

In [ ]:
df = df.dropna(subset=['quality']).copy()
df['good'] = (df['quality'] >= 6).astype(int)
X = df.drop(columns=['quality','good'])
y = df['good']
print(y.value_counts(normalize=True))

## 5. Train/test bölgüsü və preprocessing pipeline
Test məlumatını model seçimi və preprocessing fit mərhələsində istifadə etməyin. Bu nümunədə yalnız ədədi xüsusiyyətlər var; SimpleImputer və StandardScaler pipeline daxilində fit olunur.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
num_cols=X.select_dtypes(include=np.number).columns.tolist()
prep=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
preprocessor=ColumnTransformer([('num',prep,num_cols)])
print('Train:',X_train.shape,'Test:',X_test.shape)

## 6. Baseline və iki model

In [ ]:
models={
 'Dummy baseline':Pipeline([('prep',preprocessor),('model',DummyClassifier(strategy='most_frequent'))]),
 'Logistic Regression':Pipeline([('prep',preprocessor),('model',LogisticRegression(max_iter=2000,class_weight='balanced'))]),
 'Random Forest':Pipeline([('prep',preprocessor),('model',RandomForestClassifier(n_estimators=300,random_state=42,class_weight='balanced'))])
}
rows=[]
for name,m in models.items():
    m.fit(X_train,y_train)
    pred=m.predict(X_test)
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0)})
results=pd.DataFrame(rows).sort_values('F1',ascending=False)
display(results)

## 7. Confusion matrix və səhv təhlili

In [ ]:
best_name=results.iloc[0]['Model']
best=models[best_name]
pred=best.predict(X_test)
print('Seçilmiş model:',best_name)
print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,cmap='Blues');plt.title(best_name);plt.show()
errors=X_test.copy();errors['actual']=y_test;errors['predicted']=pred
errors=errors[errors.actual!=errors.predicted]
display(errors.head(10))

## 8. Cross-validation ilə sabitliyin yoxlanması
Bu addım modelin bir train/test bölünməsindən asılılığını qiymətləndirməyə kömək edir. CV nəticəsi yekun test nəticəsini əvəz etmir.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_out=cross_validate(best,X_train,y_train,cv=cv,scoring=['accuracy','f1','precision','recall'])
for metric in ['test_accuracy','test_f1','test_precision','test_recall']:
    vals=cv_out[metric]
    print(metric, 'mean=',round(vals.mean(),3),'std=',round(vals.std(),3))

## 9. Modeli saxlamaq və sadə prototip üçün hazırlamaq

In [ ]:
joblib.dump(best,'ml_pipeline.joblib')
print('Saxlanıldı: ml_pipeline.joblib')
loaded=joblib.load('ml_pipeline.joblib')
example=X_test.iloc[[0]]
print('Nümunə proqnoz:',loaded.predict(example)[0])
if hasattr(loaded,'predict_proba'):
    print('Ehtimallar:',loaded.predict_proba(example)[0])

## 10. Layihə nəticələrinin elmi şərhi
- Dataset mənbəyini, versiyasını, lisenziyasını və istifadə məhdudiyyətlərini qeyd edin.
- `quality >= 6` hədəf həddinin seçimini əsaslandırın və alternativ hədlə sensitivity analysis aparın.
- Sinif balansını nəzərə alaraq macro-F1, recall və precision-ı izah edin.
- Səhv nümunələrini yoxlayın; yüksək F1 avtomatik olaraq real tətbiqə hazır olma demək deyil.
- Bu nümunə red wine datasetində tədris üçün qurulub; başqa domenə birbaşa köçürülməməlidir.